# 06 — Null Handling

Dropping, filling, and replacing NULLs via the `na` namespace, plus null-aware functions.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Synthetic data with NULLs

In [ ]:
import pandas as pd
from irispark.functions import col, lit, coalesce, ifnull, nvl, isnull, isnotnull, when

pdf = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "nome": ["Ana", None, "Carlos", None, "Eva"],
    "valor": [100.0, 200.0, None, 400.0, 500.0],
})
df = session.createDataFrame(pdf)
df.show()

## 2. `na.drop` / `dropna`

Drop rows with any NULL (default) or all-NULL.

In [ ]:
df.na.drop().show()
df.dropna(how="all").show()
df.dropna(subset=["valor"]).show()

## 3. `na.fill` / `fillna`

Fill NULLs with a value.

In [ ]:
df.na.fill("desconhecido").show()
df.fillna(0.0, subset=["valor"]).show()

## 4. `na.replace`

Replace specific values.

In [ ]:
df.na.replace(200.0, 999.0).show()

## 5. Null-aware functions

`isnull`, `isnotnull`, `coalesce`, `ifnull`, `nvl`, `when`.

In [ ]:
df.select(
    "id", "nome", "valor",
    isnull("nome").alias("nome_isnull"),
    isnotnull("nome").alias("nome_notnull"),
    coalesce("nome", lit("N/A")).alias("nome_coalesce"),
    ifnull("valor", lit(0.0)).alias("valor_ifnull"),
    nvl("valor", lit(0.0)).alias("valor_nvl"),
    when(col("valor").isNull(), lit("missing")).otherwise(lit("ok")).alias("status"),
).show()

## 6. `eqNullSafe`

NULL-safe equality.

In [ ]:
a = session.createDataFrame(pd.DataFrame({"x": [1, None, 3]}))
b = session.createDataFrame(pd.DataFrame({"y": [1, None, 4]}))

a.join(b, a["x"].eqNullSafe(b["y"]), "inner").show()

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")